# TimesFM VN30 OHLC Volatility Comparison Test

Based on G7 Paper: Do extreme range estimators improve realized volatility forecasts?

Expected: 10-25% improvement with overnight volatility feature

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    if gpu_mem > 20:
        print("G4/L4 GPU detected - Perfect for TimesFM 2.5!")

In [ ]:
# Setup memory optimization
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print("Memory optimization enabled")

In [ ]:
# Clone repository and navigate
!git clone https://github.com/ntquy9901/stockvoli-research.git
%cd stockvoli-research
!git pull origin master
!ls -la

## Install Dependencies (OPTIONAL - Already installed in Colab)

In [ ]:
# Install dependencies (DISABLED - Already installed in Colab)
# Uncomment below if you need to install specific packages

# !pip install -q transformers peft torch pandas numpy pyyaml accelerate tqdm
# !pip install --upgrade transformers

print("Using pre-installed Colab packages (same as G4 notebook)")
print("If you get import errors, uncomment the pip install lines above")

In [ ]:
# Verify data and OHLC features
import pandas as pd
from pathlib import Path

data_dir = Path('data/processed')
files = list(data_dir.glob('*_processed.csv'))
print(f"Found {len(files)} processed stock files")
    
sample_file = files[0]
df = pd.read_csv(sample_file)
print(f"Sample: {sample_file.name}")
print(f"Shape: {df.shape}")
    
ohlc_features = ['overnight', 'parkinson', 'gk', 'close_to_close']
print(f"OHLC Features:")
for feat in ohlc_features:
    if feat in df.columns:
        count = df[feat].notna().sum()
        print(f"  {feat}: {count} observations")

In [ ]:
# Verify imports (test that all packages work)
import transformers
import peft
import torch
import pandas as pd
import numpy as np
import yaml
from tqdm import tqdm

print("All packages imported successfully!")
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

In [ ]:
# Run the OHLC comparison test
print("="*70)
print("TimesFM VN30 OHLC Volatility Comparison Test")
print("="*70)
print("")
print("Testing:")
print("  1. RV_20 (baseline) - 2 epochs (~3.7 hours)")
print("  2. Overnight volatility - 2 epochs (~3.7 hours)")
print("")
print("Expected improvement: 10-25% over baseline")
print("="*70)
print("")

# Run the test
!python src/quick_2epoch_test.py

In [ ]:
# Check results
import json
from pathlib import Path

results_file = Path('experiments/feature_comparison_results_fixed.json')
if results_file.exists():
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print("\n" + "="*70)
    print("FINAL RESULTS")
    print("="*70)
    
    if 'RV_20' in results and 'overnight' in results:
        rv20 = results['RV_20']['best_val_loss']
        overnight = results['overnight']['best_val_loss']
        improvement = (rv20 - overnight) / rv20 * 100
        
        print(f"RV_20 (baseline): {rv20:.6f}")
        print(f"Overnight volatility: {overnight:.6f}")
        print(f"Improvement: {improvement:+.1f}%")
        
        if improvement > 0:
            print("SUCCESS: Overnight volatility is better!")
            print(f"Improved forecasting by {improvement:.1f}%")
        else:
            print("Baseline still better")
else:
    print(f"Results not found: {results_file}")